# Imports

In [1]:
import numpy as np
import json
from tqdm import tqdm
import os
import shutil
import librosa
import librosa.display
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# Agrouping songs

In [3]:
name_file = "../jsons/distribution_class.json"
with open(name_file, 'r') as file:
    classes = json.load(file)

In [ ]:
source_folder = '../ICBHI_final_database'
destination_folder = '../raw_data'

for filename in tqdm(os.listdir(source_folder), desc='Agruping songs'):
    prefix = filename[:3]

    if prefix in classes:
        class_name = classes[prefix]
        
        class_folder = os.path.join(destination_folder, class_name)
        
        os.makedirs(class_folder, exist_ok=True)
        
        source_file = os.path.join(source_folder, filename)
        destination_file = os.path.join(class_folder, filename)
        
        shutil.move(source_file, destination_file)

# Generate Images

In [2]:
DATA_DIR = "../data/raw_data_merged"
OUTPUT_DIR = "../data/processed_data_merged"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SR = 44100
TARGET_DURATION = 20


def load_and_standardize(path):
    y, sr = librosa.load(path, sr=None)

    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)

    target_len = TARGET_SR * TARGET_DURATION
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    y = y / (np.max(np.abs(y)) + 1e-8)
    return y


def extract_features(y, sr):
    stft = librosa.stft(y)
    mag = np.abs(stft)

    return {
        "spec_db": librosa.amplitude_to_db(mag, ref=np.max),
        "mel_db": librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max),
        "mfcc": librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20),
        "mfcc_delta": librosa.feature.delta(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)),
        "chroma": librosa.feature.chroma_stft(S=mag, sr=sr),
        "contrast": librosa.feature.spectral_contrast(S=mag, sr=sr, fmin=50, n_bands=4),
        "cqt_db": librosa.amplitude_to_db(
            np.abs(librosa.cqt(y, sr=sr, fmin=30, n_bins=48, bins_per_octave=12)),
            ref=np.max
        ),
        "phase": np.angle(stft)
    }


def save_feature_image(feature, sr, out_path, y_axis=None):
    plt.figure(figsize=(6, 4))

    librosa.display.specshow(
        feature,
        sr=sr,
        x_axis='time',
        y_axis=y_axis
    )

    plt.axis('off')
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()


feature_configs = {
    "spec_db": {"y_axis": "log"},
    "mel_db": {"y_axis": "mel"},
    "mfcc": {"y_axis": None},
    "mfcc_delta": {"y_axis": None},
    "chroma": {"y_axis": "chroma"},
    "contrast": {"y_axis": None},
    "cqt_db": {"y_axis": "cqt_note"},
    "phase": {"y_axis": None},
}


In [3]:

for disease in os.listdir(DATA_DIR):
    disease_path = os.path.join(DATA_DIR, disease)

    if not os.path.isdir(disease_path):
        continue

    print(f"Processando: {disease}")

    for i, file in enumerate(os.listdir(disease_path)):
        if not file.endswith(".wav"):
            continue

        file_path = os.path.join(disease_path, file)
        duration = librosa.get_duration(path=file_path)
        if 18.0 < duration < 22.0:

            try:
                y = load_and_standardize(file_path)
                #TODO: fazer o y em clips de 6 segundos
                features = extract_features(y, TARGET_SR)

                base_name = os.path.splitext(file)[0]

                for feat_name, feat_value in features.items():
                    out_dir = os.path.join(OUTPUT_DIR, feat_name, disease)
                    os.makedirs(out_dir, exist_ok=True)

                    out_file = os.path.join(out_dir, f"{base_name}.png")

                    save_feature_image(
                        feat_value,
                        TARGET_SR,
                        out_file,
                        y_axis=feature_configs[feat_name]["y_axis"]
                    )

            except Exception as e:
                print(f"Erro em {file_path}: {e}")

Processando: Healthy
Processando: Pneumonia


# Generate images - 6 seconds

In [3]:
DATA_DIR = "../data/raw_data_merged"
OUTPUT_DIR = "../data/processed_data_6sec_merged"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SR = 44100
CLIP_DURATION = 6
CLIP_SAMPLES = TARGET_SR * CLIP_DURATION


def load_audio(path):
    y, sr = librosa.load(path, sr=None)

    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)

    y = y / (np.max(np.abs(y)) + 1e-8)

    return y


def split_audio_into_clips(y, clip_samples):

    total_clips = len(y) // clip_samples

    clips = []

    for i in range(total_clips):
        start = i * clip_samples
        end = start + clip_samples

        clips.append(y[start:end])

    return clips


def extract_features(y, sr):
    stft = librosa.stft(y)
    mag = np.abs(stft)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    # amplitude spectrogram to dB-scaled spectrogram.
    # Amplitude do Espectrograma na escala dB.
    return {
        "spec_db": librosa.amplitude_to_db(mag, ref=np.max),
        "mel_db": librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max),
        "mfcc": mfcc,
        "mfcc_delta": librosa.feature.delta(mfcc),
        "chroma": librosa.feature.chroma_stft(S=mag, sr=sr),
        "contrast": librosa.feature.spectral_contrast(S=mag, sr=sr, fmin=50,  n_bands=4),
        "cqt_db": librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, fmin=30, n_bins=48, bins_per_octave=12)), ref=np.max),
        "phase": np.angle(stft)
    }


def save_feature_image(feature, sr, out_path, y_axis=None):
    plt.figure(figsize=(6, 4))
    librosa.display.specshow(feature, sr=sr, x_axis='time', y_axis=y_axis)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()

feature_configs = {
    "spec_db": {"y_axis": "log"},
    "mel_db": {"y_axis": "mel"},
    "mfcc": {"y_axis": None},
    "mfcc_delta": {"y_axis": None},
    "chroma": {"y_axis": "chroma"},
    "contrast": {"y_axis": None},
    "cqt_db": {"y_axis": "cqt_note"},
    "phase": {"y_axis": None},
}


for disease in ['Healthy', 'LRTI', 'Pneumonia', 'URTI']:
# for disease in os.listdir(DATA_DIR):

    disease_path = os.path.join(DATA_DIR, disease)

    if not os.path.isdir(disease_path):
        continue

    print(f"\nProcessando: {disease}")

    for file in os.listdir(disease_path):

        if not file.endswith(".wav"):
            continue

        file_path = os.path.join(disease_path, file)

        try:
            y = load_audio(file_path)
            clips = split_audio_into_clips(y, CLIP_SAMPLES)

            if len(clips) == 0:
                print(f"Áudio muito curto ignorado: {file}")
                continue

            base_name = os.path.splitext(file)[0]

            for clip_idx, clip in enumerate(clips):

                features = extract_features(clip, TARGET_SR)

                for feat_name, feat_value in features.items():

                    out_dir = os.path.join(OUTPUT_DIR, feat_name, disease)
                    os.makedirs(out_dir, exist_ok=True)
                    out_file = os.path.join(out_dir, f"{base_name}_clip_{clip_idx:03d}.png")
                    save_feature_image(feat_value, TARGET_SR, out_file, y_axis=feature_configs[feat_name]["y_axis"])

            print(f"OK: {file} -> {len(clips)} clips")

        except Exception as e:
            print(f"Erro em {file_path}: {e}")


Processando: Healthy
OK: 102_1b1_Ar_sc_Meditron.wav -> 3 clips
OK: 121_1b1_Tc_sc_Meditron.wav -> 3 clips
OK: 121_1p1_Tc_sc_Meditron.wav -> 3 clips
OK: 123_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 125_1b1_Tc_sc_Meditron.wav -> 3 clips
OK: 126_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 127_1b1_Ar_sc_Meditron.wav -> 3 clips
OK: 136_1b1_Ar_sc_Meditron.wav -> 3 clips
OK: 143_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 144_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 144_1b1_Tc_sc_Meditron.wav -> 3 clips
OK: 152_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 153_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 159_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 159_1b1_Ar_sc_Meditron.wav -> 3 clips
OK: 159_1b1_Ll_sc_Meditron.wav -> 3 clips
OK: 159_1b1_Pr_sc_Meditron.wav -> 3 clips
OK: 171_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 179_1b1_Al_sc_Meditron.wav -> 3 clips
OK: 179_1b1_Tc_sc_Meditron.wav -> 3 clips
OK: 182_1b1_Tc_sc_Meditron.wav -> 3 clips
OK: 183_1b1_Pl_sc_Meditron.wav -> 3 clips
OK: 183_1b1_Tc_sc_Meditron.wav -> 3 clips
OK: 184_1b1_